# <center> <font color="#0036a3">Maestría en Inteligencia Artificial Aplicada (MNA)</font> </center>

<center>

[![Materia](https://img.shields.io/badge/MATERIA-PROYECTO_INTEGRADOR-E0A800?style=for-the-badge&logoColor=white)](https://tec.mx)

</center>

<center>

[![Python](https://img.shields.io/badge/Python-3776AB?style=flat-square&logo=python&logoColor=white)](https://www.python.org/)
[![Jupyter](https://img.shields.io/badge/Jupyter-F37626?style=flat-square&logo=jupyter&logoColor=white)](https://jupyter.org/)
[![PyTorch](https://img.shields.io/badge/PyTorch-EE4C2C?style=flat-square&logo=pytorch&logoColor=white)](https://pytorch.org/)
[![OpenCV](https://img.shields.io/badge/OpenCV-5C3EE8?style=flat-square&logo=opencv&logoColor=white)](https://opencv.org/)
[![GitHub](https://img.shields.io/badge/Repo-GitHub-181717?style=flat-square&logo=github&logoColor=white)](https://github.com/jmtoral/proyecto_integrador_52)

</center>

## **<font color="#0036a3">Avance 4 — Evaluación de Image Enhancement (Retinex, EndoLMSPEC e IAT/EndoViT) sobre Múltiples Modelos de Profundidad Endoscópica en SCARED</font>**

### **<font color="#E0A800">Proyecto Integrador — TC5035.10</font>**

---

## **<center> <font color="#0036a3">Equipo 52</font> </center>**

<table style="border-collapse:collapse; width:60%; margin:auto;">
  <tr>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/elda.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>Elda Morales</strong><br><small>A00449074</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/mpgc.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>María Paula Gutiérrez</strong><br><small>A01747706</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/jmtc_n.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>José Manuel Toral</strong><br><small>A01122243</small>
    </td>
  </tr>
</table>

---

### Objetivos de este notebook

Evaluar si los modelos de **Image Enhancement** mejoran el desempeño de **múltiples modelos de depth estimation** sobre SCARED:

1. Comparar **4 métodos de enhancement**: None (baseline), Retinex SSR, EndoLMSPEC e IAT/EndoViT
2. Evaluar sobre **2 modelos de depth**: Endo-Depth (Recasens et al., 2021) y EndoSfMLearner (Ozyoruk et al., 2020)
3. Cuantificar el impacto en: **AbsRel, RMSE, Chamfer Distance**
4. Analizar si la mejora se concentra en **zonas especulares** (hipótesis central)
5. Evaluar **viabilidad clínica** (FPS)

> García-Vega, A., et al. (2022). Multi-Scale Structural-aware Exposure Correction for Endoscopic Imaging. *arXiv:2210.15033*.
> Rahman, Z., et al. (2004). Retinex processing for automatic image enhancement. *Journal of Electronic Imaging*, 13(1). https://doi.org/10.1117/1.1636183
> Wang, T., et al. (2022). Ultra-High-Definition Low-Light Image Enhancement. *AAAI 2022*.
> Ozyoruk, K.B., et al. (2020). EndoSLAM Dataset and Endo-SfMLearner. *arXiv:2006.16670*.

---
## 0. Pipeline general del experimento

```mermaid
flowchart TD
    A["🗂️ SCARED\ndatasets 8–9, keyframes 0–4"] --> B
    B["📷 Imagen RGB\n1280×1024 px"] --> C1 & C2 & C3 & C4

    subgraph ENHANCEMENT [" ⚙️ Image Enhancement "]
        C1["❌ None"]
        C2["🔆 Retinex SSR\nRahman et al. 2004"]
        C3["🧠 EndoLMSPEC\nEndo4IE\nGarcía-Vega et al. 2022"]
        C4["⚡ IAT — EndoViT\nEndo4IE\nWang et al. 2022"]
    end

    C1 & C2 & C3 & C4 --> D1 & D2 & D3

    subgraph DEPTH [" 🔍 Modelos de Depth "]
        D1["Endo-Depth\nResNet18\nRecasens et al. 2021"]
        D2["EndoSfMLearner\nDispResNet18\nOzyoruk et al. 2020"]
        D3["MonoViT\nMPViT-Small\nZhao et al. 2022"]
    end

    D1 & D2 & D3 --> E["📐 Median Scaling → mm"]
    E --> F1 & F2 & F3 & F4 & F5 & F6

    subgraph METRICS [" 📊 Métricas "]
        F1["AbsRel"]
        F2["RMSE mm"]
        F3["Chamfer mm"]
        F4["AbsRel especular"]
        F5["PSNR / SSIM"]
        F6["FPS"]
    end

    style ENHANCEMENT fill:#fff8e1,stroke:#E0A800,stroke-width:2px
    style DEPTH fill:#e8f4fd,stroke:#2766CB,stroke-width:2px
    style METRICS fill:#f0f0f0,stroke:#888,stroke-width:1px
```

**Diseño factorial**: 4 enhancements × 3 modelos de depth × 10 keyframes = **120 evaluaciones**

### Hipótesis
Los métodos de enhancement entrenados en datos endoscópicos (EndoLMSPEC, IAT/EndoViT) reducen el error de profundidad en los tres modelos, con mayor efecto en zonas especulares.

In [1]:
import subprocess, sys

# Solo lo que Colab no trae por defecto
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tifffile", "scikit-image"])

import torch, tifffile, cv2, numpy as np
print(f"torch    : {torch.__version__}")
print(f"tifffile : {tifffile.__version__}")
print(f"opencv   : {cv2.__version__}")
print(f"CUDA OK  : {torch.cuda.is_available()}")

torch    : 2.11.0+cu128
tifffile : 2026.4.11
opencv   : 4.13.0
CUDA OK  : True


---
## 1. Configuración de rutas

### Estructura esperada en Google Drive

```
MyDrive/proyecto_integrador/
├── scared_raw/
├── endo_depth_weights/            ← encoder.pth + depth.pth  (Endo-Depth)
├── Endo-Depth-and-Motion/
├── EndoLMSPEC/
│   └── checkpoint/main_net/model_256_combined_SSIM5_1.pth
├── EndoViT/
│   └── Endo4IE/best_Epoch50_laplacian_histogan_loss.pth
├── EndoSLAM/
│   ├── EndoSfMLearner/
│   └── pretrained/08-13-00_00/dispnet_model_best.pth.tar
├── MonoViT/
│   ├── networks/                  ← mpvit.py (parcheado), hr_decoder.py
│   └── mono_640x192/encoder.pth + depth.pth
└── avance4_outputs/
```

### Modelos de depth — resumen de pesos

| Modelo | Archivo(s) de pesos | Backbone | Dataset entrenamiento | Resolución |
|---|---|---|---|---|
| **Endo-Depth** | `endo_depth_weights/encoder.pth` + `depth.pth` | ResNet-18 | Hamlyn (laparoscopy) | 640×192 |
| **EndoSfMLearner** | `EndoSLAM/pretrained/08-13-00_00/dispnet_model_best.pth.tar` | ResNet-18 (DispResNet) | EndoSLAM (capsule endo.) | 256×832 |
| **MonoViT** | `MonoViT/mono_640x192/encoder.pth` + `depth.pth` | MPViT-Small (ViT) | KITTI (outdoor) | 640×192 |

### Métodos de enhancement — resumen de pesos

| Método | Archivo de pesos | Arquitectura | Dataset entrenamiento | Parámetros |
|---|---|---|---|---|
| **Retinex SSR** | — (sin pesos) | Filtro gaussiano logarítmico | — | 0 |
| **EndoLMSPEC** | `EndoLMSPEC/checkpoint/main_net/model_256_combined_SSIM5_1.pth` | U-Net multi-escala + pirámide Laplaciana | Endo4IE (over+under) | ~2 M |
| **IAT / EndoViT** | `EndoViT/Endo4IE/best_Epoch50_laplacian_histogan_loss.pth` | Transformer local+global (IAT) | Endo4IE fine-tune, época 50 | ~90 K |

In [ ]:
from pathlib import Path
import sys

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE            = Path("/content/drive/MyDrive/proyecto_integrador")
    MODEL_PATH      = BASE / "endo_depth_weights"
    SCARED_ROOT     = BASE / "scared_raw"
    EDAM_PATH       = BASE / "Endo-Depth-and-Motion"
    LMSPEC_PATH     = BASE / "EndoLMSPEC"
    IAT_PATH        = BASE / "EndoViT"
    ENDOSLAM_PATH   = BASE / "EndoSLAM"
    MONOVIT_PATH    = BASE / "MonoViT"
    OUT_DIR         = BASE / "avance4_outputs"
else:
    MODEL_PATH      = Path("E:/endo_depth_weights")
    SCARED_ROOT     = Path("D:/Proyecto_Integrador/Corrreccion_Luz/data/scared_raw")
    EDAM_PATH       = Path("E:/Endo-Depth-and-Motion")
    LMSPEC_PATH     = Path("E:/EndoLMSPEC")
    IAT_PATH        = Path("E:/EndoVit")
    ENDOSLAM_PATH   = Path("E:/EndoSLAM")
    MONOVIT_PATH    = Path("E:/MonoViT")
    OUT_DIR         = Path("../outcomes/avance4_enhancement")

LMSPEC_WEIGHTS   = LMSPEC_PATH / "checkpoint" / "main_net" / "model_256_combined_SSIM5_1.pth"
IAT_WEIGHTS      = IAT_PATH / "Endo4IE" / "best_Epoch50_laplacian_histogan_loss.pth"
ENDOSFM_WEIGHTS  = ENDOSLAM_PATH / "pretrained" / "08-13-00_00" / "dispnet_model_best.pth.tar"
MONOVIT_WEIGHTS  = MONOVIT_PATH / "mono_640x192"   # carpeta con encoder.pth + depth.pth

OUT_DIR.mkdir(parents=True, exist_ok=True)

EVAL_KEYFRAMES = [
    ("dataset_8", "keyframe_0"), ("dataset_8", "keyframe_1"),
    ("dataset_8", "keyframe_2"), ("dataset_8", "keyframe_3"),
    ("dataset_8", "keyframe_4"), ("dataset_9", "keyframe_0"),
    ("dataset_9", "keyframe_1"), ("dataset_9", "keyframe_2"),
    ("dataset_9", "keyframe_3"), ("dataset_9", "keyframe_4"),
]
CAP_MM = 150.0

print(f"Entorno : {'Google Colab' if IN_COLAB else 'Local'}")
for name, p in [("MODEL_PATH", MODEL_PATH), ("SCARED_ROOT", SCARED_ROOT),
                ("LMSPEC_WEIGHTS", LMSPEC_WEIGHTS), ("IAT_WEIGHTS", IAT_WEIGHTS),
                ("ENDOSFM_WEIGHTS", ENDOSFM_WEIGHTS), ("MONOVIT_WEIGHTS", MONOVIT_WEIGHTS)]:
    status = "✓" if p.exists() else "✗ NO ENCONTRADO"
    print(f"  {name:18s}: {status}")

---
## 2. Cargar Endo-Depth

Mismo modelo que en Avances 3 y 4: ResNet18 encoder + DepthDecoder, pesos Hamlyn. No se modifica el modelo — es la variable dependiente fija del experimento.

In [3]:
import torch
import numpy as np

sys.path.insert(0, str(EDAM_PATH / "apps" / "depth_estimate"))
from resnet_encoder import ResnetEncoder
from depth_decoder import DepthDecoder

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")

encoder = ResnetEncoder(18, False)
loaded_enc = torch.load(MODEL_PATH / "encoder.pth", map_location=DEVICE)
FEED_HEIGHT = loaded_enc["height"]
FEED_WIDTH  = loaded_enc["width"]
filtered_enc = {k: v for k, v in loaded_enc.items() if k in encoder.state_dict()}
encoder.load_state_dict(filtered_enc)
encoder.to(DEVICE).eval()

depth_decoder = DepthDecoder(num_ch_enc=encoder.num_ch_enc, scales=range(4))
loaded_dec = torch.load(MODEL_PATH / "depth.pth", map_location=DEVICE)
depth_decoder.load_state_dict(loaded_dec)
depth_decoder.to(DEVICE).eval()

print(f"Resolución del modelo: {FEED_HEIGHT}×{FEED_WIDTH}")
print("Endo-Depth cargado ✓")

Dispositivo: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Resolución del modelo: 256×320
Endo-Depth cargado ✓


In [4]:
import sys
import torch
import torch.nn.functional as F
import numpy as np

# Agregar EndoSfMLearner al path
sys.path.insert(0, str(ENDOSLAM_PATH / "EndoSfMLearner"))
import models as endosfm_models

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DispResNet con ResNet-18 backbone
endosfm_net = endosfm_models.DispResNet(18, False).to(DEVICE)
weights = torch.load(ENDOSFM_WEIGHTS, map_location=DEVICE)
endosfm_net.load_state_dict(weights['state_dict'])
endosfm_net.eval()

n_params = sum(p.numel() for p in endosfm_net.parameters())
print(f"EndoSfMLearner parámetros: {n_params/1e6:.2f} M")
print(f"Pesos: {ENDOSFM_WEIGHTS.name}")
print("EndoSfMLearner cargado ✓")

EndoSfMLearner parámetros: 14.84 M
Pesos: dispnet_model_best.pth.tar
EndoSfMLearner cargado ✓


---
## 2b. EndoSfMLearner — Endo-SfMLearner (Ozyoruk et al., 2020)

**EndoSfMLearner** es un método de estimación de profundidad monocular auto-supervisado específicamente diseñado para endoscopía, publicado como parte del dataset EndoSLAM. Sus contribuciones principales son:

- **Brightness-aware photometric loss**: hace la predicción de profundidad robusta a variaciones de iluminación — relevante directamente para nuestra hipótesis de corrección de iluminación
- **Spatial attention-based pose network**: optimizado para las características geométricas de imágenes endoscópicas

La arquitectura usa **DispResNet** (ResNet-18 como backbone), misma familia que Endo-Depth, pero con una cabeza de disparidad diferente y entrenado en datos de endoscopía de cápsula (EndoSLAM dataset).

> Ozyoruk, K. B., et al. (2020). EndoSLAM Dataset and An Unsupervised Monocular Visual Odometry and Depth Estimation Approach for Endoscopic Videos: Endo-SfMLearner. *arXiv:2006.16670*.

In [5]:
import sys
import torch
import numpy as np

# timm y einops son necesarios para MPViT
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm", "einops"])

# Agregar MonoViT al path — mpvit.py ya fue parcheado (sin mmcv/mmseg)
sys.path.insert(0, str(MONOVIT_PATH))
from networks.nets import DeepNet

# DeepNet encapsula encoder + decoder juntos (no separarlos)
monovit_net = DeepNet(type='mpvitnet')

# Cargar pesos del encoder (contiene height/width de entrenamiento)
enc_weights = torch.load(MONOVIT_WEIGHTS / "encoder.pth", map_location=DEVICE)
MONOVIT_H = enc_weights.get("height", 192)
MONOVIT_W = enc_weights.get("width",  640)

# Cargar encoder en monovit_net.encoder
filtered_enc = {k.replace("encoder.", ""): v
                for k, v in enc_weights.items()
                if k.replace("encoder.", "") in monovit_net.encoder.state_dict()}
monovit_net.encoder.load_state_dict(filtered_enc, strict=False)

# Cargar decoder en monovit_net.decoder
dec_weights = torch.load(MONOVIT_WEIGHTS / "depth.pth", map_location=DEVICE)
monovit_net.decoder.load_state_dict(dec_weights)

monovit_net.to(DEVICE).eval()

# Aliases para compatibilidad con predict_depth_monovit
monovit_enc = monovit_net.encoder
monovit_dec = monovit_net.decoder

n_params = sum(p.numel() for p in monovit_net.parameters())
print(f"MonoViT parámetros: {n_params/1e6:.1f} M")
print(f"Resolución        : {MONOVIT_H}×{MONOVIT_W}")
print("MonoViT cargado ✓")

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


MonoViT parámetros: 27.9 M
Resolución        : 192×640
MonoViT cargado ✓


---
## 2c. MonoViT — Self-Supervised Monocular Depth with Vision Transformer (Zhao et al., 2022)

**MonoViT** reemplaza el encoder ResNet de Monodepth2 con **MPViT-Small** (*Multi-Path Vision Transformer*), un transformer que procesa parches de imagen en múltiples escalas simultáneamente. El decoder es `HR-Depth` (high-resolution decoder con attention modules).

Sus ventajas clave para este experimento:
- **Transformer encoder**: captura dependencias de largo rango en la imagen — potencialmente más robusto a variaciones de iluminación localizada (especulares) que los encoders convolucionales
- **Multi-path**: procesa la imagen a 4 escalas en paralelo, combinando contexto global y detalle local
- **Estado del arte** en KITTI (AbsRel=0.099 @ 640×192)

> Zhao, C., Zhang, Y., Poggi, M., Tosi, F., Guo, X., Zhu, Z., Huang, G., Tang, Y., & Mattoccia, S. (2022). MonoViT: Self-Supervised Monocular Depth Estimation with a Vision Transformer. *3DV 2022*. arXiv:2208.03543.

---
## 3. EndoLMSPEC — Arquitectura y carga del modelo

### 3.1 ¿Qué es EndoLMSPEC?

EndoLMSPEC (*Multi-Scale Structural-aware Exposure Correction for Endoscopic Imaging*, García-Vega et al., 2022) es una extensión del método LMSPEC de Afifi et al. (2021) adaptada específicamente para imágenes endoscópicas. Resuelve el problema de **corrección de exposición** (tanto sobreexposición por reflejos especulares como subexposición en zonas periféricas) sin datos pareados anotados manualmente — aprende a corregir usando la distribución estadística de imágenes endoscópicas normales.

Fue entrenado en **Endo4IE**, el primer dataset endoscópico diseñado específicamente para evaluación de métodos de image enhancement, que contiene frames reales y sintéticos con exposición incorrecta y sus correspondientes imágenes de referencia.

### 3.2 Arquitectura detallada

```mermaid
flowchart LR
    IN["Imagen de entrada\nHxWx3 float32"] --> LP

    subgraph LP [" 🔺 Pirámide Laplaciana (4 niveles) "]
        direction TB
        L4["Level 4\nH/8 × W/8\n(low-freq)"]
        L3["Level 3\nH/4 × W/4"]
        L2["Level 2\nH/2 × W/2"]
        L1["Level 1\nH × W\n(high-freq)"]
    end

    L4 --> U1

    subgraph UNETS [" 🧱 Cascada de U-Nets "]
        direction TB
        U1["UNet24\n(sin residual)\nCorrige low-freq"]
        U2["UNet24-res\n(+ residual)\nRefina freq. medias"]
        U3["UNet24-res\n(+ residual)\nRefina más detalle"]
        U4["UNet16-res\n(+ residual)\nReconstrucción final"]
    end

    U1 -->|"y_hat0 + L3"| U2
    U2 -->|"y_hat1 + L2"| U3
    U3 -->|"y_hat2 + L1"| U4

    U4 --> OUT["subnet_16\nImagen corregida\nHxWx3"]

    style LP fill:#fff8e1,stroke:#E0A800
    style UNETS fill:#e8f4fd,stroke:#2766CB
```

**Clave de la arquitectura**: la imagen de entrada se descompone en una **pirámide Laplaciana** de 4 niveles. El nivel 4 (más pequeño, baja frecuencia) captura la iluminación global; los niveles superiores capturan detalles de alta frecuencia (texturas, bordes, venas). Cada U-Net procesa un nivel de la pirámide y pasa su resultado al siguiente nivel sumando con los detalles de alta frecuencia. Esto permite corregir la iluminación global sin destruir la textura del tejido.

### 3.3 Pirámide Laplaciana

La pirámide Laplaciana descompone la imagen como:

$$L_k = G_k - \text{pyrUp}(G_{k+1})$$

donde $G_k$ es el nivel $k$ de la pirámide Gaussiana. Cada $L_k$ contiene las **diferencias** entre escalas consecutivas — esencialmente el contenido de alta frecuencia a esa escala. La imagen original se puede reconstruir exactamente sumando todos los niveles de la pirámide Laplaciana.

**¿Por qué es ideal para endoscopía?** Los reflejos especulares son eventos de alta frecuencia localizada (brillos puntuales), mientras que el gradiente radial de iluminación del endoscopio es de baja frecuencia. La pirámide Laplaciana los separa naturalmente, permitiendo que la red corrija la iluminación global (niveles bajos) sin alterar los detalles del tejido (niveles altos).

### 3.4 Función de pérdida

EndoLMSPEC usa una pérdida compuesta de 4 términos:

$$\mathcal{L} = \alpha \mathcal{L}_{rec} + \beta \mathcal{L}_{pyr} + \gamma \mathcal{L}_{SSIM} + \delta \mathcal{L}_{adv}$$

donde:
- $\mathcal{L}_{rec}$: reconstrucción pixel-wise (L1)
- $\mathcal{L}_{pyr}$: consistencia de pirámide Laplaciana
- $\mathcal{L}_{SSIM}$: similitud estructural (aportación principal vs. LMSPEC original)
- $\mathcal{L}_{adv}$: pérdida adversarial (discriminador)

La inclusión de $\mathcal{L}_{SSIM}$ es la innovación principal respecto al LMSPEC original: fuerza a la red a preservar la **estructura del tejido** (bordes, venas, pliegues) incluso al corregir la exposición — esencial para que la imagen corregida sea útil para estimación de profundidad.

In [ ]:
import sys
import subprocess
import torch

# Instalar dependencia que EndoSLAM/utils.py necesita
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "path"])

# EndoLMSPEC tiene conflicto con EndoSLAM/utils.py en sys.path.
# Cargamos generator.py con un sys.path temporal limpio.
import importlib.util, types

def _load_endolmspec(lmspec_path, device):
    """Carga Generator aislando su path para evitar conflictos."""
    # Guardar y limpiar sys.path temporalmente
    _orig_path = sys.path.copy()
    # Poner EndoLMSPEC PRIMERO y quitar EndoSLAM del path
    clean = [str(lmspec_path)] + [
        p for p in sys.path
        if "EndoSLAM" not in p and "endosfm" not in p.lower()
    ]
    sys.modules.pop("utils", None)
    sys.modules.pop("generator", None)
    sys.modules.pop("unet", None)

    try:
        sys.path = clean
        spec = importlib.util.spec_from_file_location(
            "generator", lmspec_path / "generator.py")
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        Generator = mod.Generator
    finally:
        sys.path = _orig_path

    net = Generator(n_channels=3, device=device, bilinear=False)
    return net, Generator

lmspec_net, _ = _load_endolmspec(LMSPEC_PATH, DEVICE)
lmspec_net.load_state_dict(torch.load(LMSPEC_WEIGHTS, map_location=DEVICE))
lmspec_net.to(DEVICE).eval()

n_params = sum(p.numel() for p in lmspec_net.parameters())
print(f"EndoLMSPEC parámetros: {n_params/1e6:.2f} M")
print(f"Pesos: {LMSPEC_WEIGHTS.name}")
print("EndoLMSPEC cargado ✓")

---
## 3b. IAT / EndoViT — Illumination-Adaptive Transformer fine-tuneado en Endo4IE

### ¿Qué es IAT y cómo se relaciona con EndoViT?

El repositorio **EndoViT** (proporcionado por el Dr. Gilberto Ochoa Ruiz) contiene una implementación del **IAT** (*Illumination-Adaptive Transformer*, Wang et al., 2022) con pesos fine-tuneados en el dataset **Endo4IE** — específicamente con una función de pérdida que combina pirámide Laplaciana e histograma GAN (`best_Epoch50_laplacian_histogan_loss.pth`). Esto lo distingue del IAT original entrenado en imágenes naturales (MIT-FiveK) y lo hace directamente comparable con EndoLMSPEC, que también fue entrenado en Endo4IE.

IAT opera en dos ramas paralelas:

- **Rama local** (`Local_pred_S`): predice dos mapas por píxel — un factor multiplicativo `mul` y un desplazamiento aditivo `add`: $\hat{I}_{local} = I \odot mul + add$
- **Rama global** (`Global_pred`): un Swin Transformer que predice gamma $\gamma$ y una matriz de corrección de color CCM 3×3

```mermaid
flowchart LR
    IN["Imagen\n[0,1] float\n(Endo4IE fine-tune)"] --> LP & GP

    subgraph LP [" 🔲 Rama Local "]
        LC["Conv + 3×CBlock_ln"] --> MUL["mul map\n(ReLU)"]
        LC --> ADD["add map\n(Tanh)"]
    end

    subgraph GP [" 🌐 Rama Global "]
        ST["Swin Transformer\n(Global_pred)"] --> GAMMA["γ (gamma)"]
        ST --> CCM["CCM 3×3\n(color matrix)"]
    end

    IN --> MULT["I × mul + add\n= I_local"]
    MUL & ADD --> MULT
    MULT --> APPLY["apply_color(I_local, CCM)^γ"]
    GAMMA & CCM --> APPLY
    APPLY --> OUT["Imagen corregida\n[0,1] float"]

    style LP fill:#e8f4fd,stroke:#2766CB
    style GP fill:#fff8e1,stroke:#E0A800
```

### Pesos utilizados

| Archivo | Descripción |
|---|---|
| `best_Epoch50_laplacian_histogan_loss.pth` | Fine-tune en Endo4IE (over+underexp), pérdida Laplaciana + HistoGAN, epoch 50 |

Otros checkpoints disponibles en `EndoVit/Endo4IE/`: epoch 10, 20 con variantes de normalización.

> Wang, T., Zhang, K., Shen, T., Luo, W., Stenger, B., & Lu, T. (2022). Ultra-High-Definition Low-Light Image Enhancement: A Benchmark and Transformer-Based Method. *AAAI 2022*.

In [ ]:
import sys
import types
import importlib.util
import torch

# global_net.py tiene `import imp` que fue removido en Python 3.12.
# imp no se usa en el código — solo está importado. Inyectamos un módulo
# ficticio para que el import no falle.
sys.modules['imp'] = types.ModuleType('imp')

# Importar IAT directamente por path para evitar conflicto con
# el módulo 'model' de Endo-Depth ya cargado en sys.path
iat_model_path = IAT_PATH / "experiments" / "model" / "IAT_main.py"
spec = importlib.util.spec_from_file_location("IAT_main", iat_model_path)
iat_module = importlib.util.module_from_spec(spec)

_iat_exp_path = str(IAT_PATH / "experiments")
if _iat_exp_path not in sys.path:
    sys.path.insert(0, _iat_exp_path)

spec.loader.exec_module(iat_module)
IAT = iat_module.IAT

# type='exp' — configuración usada en train_exposure.py
iat_net = IAT(in_dim=3, with_global=True, type='exp')
iat_net.load_state_dict(torch.load(IAT_WEIGHTS, map_location=DEVICE))
iat_net.to(DEVICE).eval()

n_params = sum(p.numel() for p in iat_net.parameters())
print(f"IAT parámetros : {n_params/1e3:.1f} K")
print(f"Pesos          : {IAT_WEIGHTS.name}")
print("IAT (EndoViT) cargado ✓")

---
## 4. Métodos de Image Enhancement

### 4.1 Baseline: sin corrección
La imagen original sin ningún preprocesamiento. Referencia para cuantificar el impacto de los métodos.

### 4.2 Retinex SSR
$$R_{SSR}(x,y) = \log I(x,y) - \log\left[G_\sigma * I(x,y)\right], \quad \sigma=30$$
> Rahman, Z., Jobson, D. J., & Woodell, G. A. (2004). *Journal of Electronic Imaging*, 13(1), 100–110. https://doi.org/10.1117/1.1636183

### 4.3 EndoLMSPEC
Pirámide Laplaciana de 4 niveles → cascada de 4 U-Nets con pérdida SSIM. Salida: `subnet_16` a resolución completa. Ver sección 3 para detalles.

### 4.4 IAT (Illumination-Adaptive Transformer)
Corrección local (mul/add por píxel vía conv) + corrección global (gamma + CCM vía Swin Transformer). ~90K parámetros, muy rápido. Ver sección 3b para detalles.

> Wang, T., et al. (2022). Ultra-High-Definition Low-Light Image Enhancement. *AAAI 2022*.

In [ ]:
import cv2
import numpy as np
import torch
import torchvision.transforms as T


def correct_none(img_rgb: np.ndarray) -> np.ndarray:
    return img_rgb


def correct_retinex(img_rgb: np.ndarray, sigma: float = 30) -> np.ndarray:
    """Single-Scale Retinex (Rahman et al., 2004)."""
    img_f = img_rgb.astype(np.float32) + 1.0
    result = np.zeros_like(img_f)
    for c in range(3):
        blur = cv2.GaussianBlur(img_f[:, :, c], (0, 0), sigma)
        result[:, :, c] = np.log(img_f[:, :, c]) - np.log(blur + 1.0)
    result -= result.min()
    return (result / (result.max() + 1e-8) * 255).astype(np.uint8)


def correct_endolmspec(img_rgb: np.ndarray, net, device) -> np.ndarray:
    """EndoLMSPEC: pirámide Laplaciana + U-Nets (García-Vega et al., 2022)."""
    img_t = T.ToTensor()(img_rgb).to(device)
    with torch.no_grad():
        _, outputs = net(img_t)
    out = outputs['subnet_16'][0].cpu().clamp(0, 1)
    return (out.permute(1, 2, 0).numpy() * 255).astype(np.uint8)


def correct_iat(img_rgb: np.ndarray, net, device) -> np.ndarray:
    """IAT: local mul/add + global gamma+CCM (Wang et al., 2022)."""
    # IAT espera [0,1] float, sin normalización adicional
    img_f = img_rgb.astype(np.float32) / 255.0
    img_t = torch.from_numpy(img_f).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        _, _, enhanced = net(img_t)
    out = enhanced[0].cpu().clamp(0, 1)
    return (out.permute(1, 2, 0).numpy() * 255).astype(np.uint8)


CORRECTIONS = {
    "none":       lambda img: correct_none(img),
    "retinex":    lambda img: correct_retinex(img),
    "endolmspec": lambda img: correct_endolmspec(img, lmspec_net, DEVICE),
    "iat":        lambda img: correct_iat(img, iat_net, DEVICE),
}

print("Funciones de corrección definidas ✓")
print(f"Métodos: {list(CORRECTIONS.keys())}")

---
## 5. Carga de datos SCARED

Mismo protocolo que Avance 3: datasets 8–9 (split de test, animal distinto al de train). Cada keyframe contiene `Left_Image.png` y `left_depth_map.tiff` (canal Z en mm).

In [ ]:
import time
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import PIL.Image as pil
from torchvision import transforms
from scipy.spatial import cKDTree
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.transform import resize as imresize

FX, FY = 1078.0, 1078.0
CX, CY = 640.0,  512.0


def predict_depth_endodepth(img_rgb, encoder, decoder, feed_h, feed_w, device):
    H, W = img_rgb.shape[:2]
    input_t = transforms.ToTensor()(
        pil.fromarray(img_rgb).resize((feed_w, feed_h), pil.LANCZOS)
    ).unsqueeze(0).to(device)
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        features = encoder(input_t)
        outputs  = decoder(features)
    if device.type == "cuda": torch.cuda.synchronize()
    t_ms = (time.perf_counter() - t0) * 1000
    disp = F.interpolate(outputs[("disp", 0)], (H, W), mode="bilinear", align_corners=False)
    disp_np = disp.squeeze().cpu().numpy()
    min_d, max_d = 1/100, 1/0.1
    return 1.0 / (min_d + (max_d - min_d) * disp_np), t_ms


def predict_depth_endosfm(img_rgb, net, device, img_h=256, img_w=832):
    H, W = img_rgb.shape[:2]
    img_r = imresize(img_rgb, (img_h, img_w)).astype(np.float32)
    img_t = torch.from_numpy(
        ((img_r / 255.0 - 0.45) / 0.225).transpose(2, 0, 1)
    ).unsqueeze(0).to(device)
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        disp = net(img_t)
    if device.type == "cuda": torch.cuda.synchronize()
    t_ms = (time.perf_counter() - t0) * 1000
    disp_full = imresize(disp.squeeze().cpu().numpy(), (H, W))
    return 1.0 / (disp_full + 1e-6), t_ms


def predict_depth_monovit(img_rgb, net, feed_h, feed_w, device):
    """MonoViT: usa DeepNet completo (enc+dec juntos). Norm: ImageNet."""
    H, W = img_rgb.shape[:2]
    input_t = transforms.ToTensor()(
        pil.fromarray(img_rgb).resize((feed_w, feed_h), pil.LANCZOS)
    ).unsqueeze(0).to(device)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(device)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(device)
    input_t = (input_t - mean) / std
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = net(input_t)   # DeepNet.forward() llama enc→dec internamente
    if device.type == "cuda": torch.cuda.synchronize()
    t_ms = (time.perf_counter() - t0) * 1000
    disp = F.interpolate(outputs[("disp", 0)], (H, W), mode="bilinear", align_corners=False)
    disp_np = disp.squeeze().cpu().numpy()
    min_d, max_d = 1/100, 1/0.1
    return 1.0 / (min_d + (max_d - min_d) * disp_np), t_ms


# Verificar que MonoViT cargó antes de definir DEPTH_MODELS
try:
    _ = monovit_net
    monovit_ok = True
except NameError:
    monovit_ok = False
    print("⚠️  MonoViT NO cargado — corre cell-9 primero")

DEPTH_MODELS = {
    "EndoDepth":      lambda img: predict_depth_endodepth(
                          img, encoder, depth_decoder, FEED_HEIGHT, FEED_WIDTH, DEVICE),
    "EndoSfMLearner": lambda img: predict_depth_endosfm(img, endosfm_net, DEVICE),
}
if monovit_ok:
    DEPTH_MODELS["MonoViT"] = lambda img: predict_depth_monovit(
                                   img, monovit_net, MONOVIT_H, MONOVIT_W, DEVICE)


def depth_to_pc(depth_mm, mask, fx=FX, fy=FY, cx=CX, cy=CY):
    H, W = depth_mm.shape
    uu, vv = np.meshgrid(np.arange(W), np.arange(H))
    Z = depth_mm[mask]
    return np.stack([(uu[mask]-cx)*Z/fx, (vv[mask]-cy)*Z/fy, Z], axis=1)

def chamfer(pc1, pc2, max_pts=50_000):
    if len(pc1) == 0 or len(pc2) == 0: return np.nan
    rng = np.random.default_rng(42)
    if len(pc1) > max_pts: pc1 = pc1[rng.choice(len(pc1), max_pts, replace=False)]
    if len(pc2) > max_pts: pc2 = pc2[rng.choice(len(pc2), max_pts, replace=False)]
    d1, _ = cKDTree(pc2).query(pc1)
    d2, _ = cKDTree(pc1).query(pc2)
    return float((d1.mean() + d2.mean()) / 2)

def specular_mask(img_rgb, pct=97, dil=15):
    L = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)[:,:,0].astype(np.float32)
    mask = (L >= np.percentile(L, pct)).astype(np.uint8)
    return cv2.dilate(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(dil,dil))).astype(bool)

def compute_metrics(img_orig, img_corr, depth_rel, gt_mm, cap_mm=150.0):
    valid = (~np.isnan(gt_mm)) & (gt_mm > 0) & (gt_mm < cap_mm)
    if valid.sum() == 0:
        return {k: np.nan for k in
                ["AbsRel","RMSE","Chamfer","PSNR","SSIM","AbsRel_spec","AbsRel_nospec","scale"]}
    scale = np.median(gt_mm[valid]) / (np.median(depth_rel[valid]) + 1e-8)
    pred  = depth_rel * scale
    d, gt = pred[valid], gt_mm[valid]
    spec  = specular_mask(img_orig)
    vs, vn = valid & spec, valid & ~spec
    return {
        "scale":         round(float(scale), 4),
        "AbsRel":        round(float(np.mean(np.abs(d-gt)/(gt+1e-8))), 4),
        "RMSE":          round(float(np.sqrt(np.mean((d-gt)**2))), 2),
        "Chamfer":       round(chamfer(depth_to_pc(np.where(valid,pred,np.nan),valid),
                                       depth_to_pc(np.where(valid,gt_mm,np.nan),valid)), 3),
        "PSNR":          round(float(psnr_fn(img_orig, img_corr, data_range=255)), 2),
        "SSIM":          round(float(ssim_fn(img_orig, img_corr, channel_axis=2, data_range=255)), 4),
        "AbsRel_spec":   round(float(np.mean(np.abs(pred[vs]-gt_mm[vs])/(gt_mm[vs]+1e-8))),4) if vs.sum()>0 else np.nan,
        "AbsRel_nospec": round(float(np.mean(np.abs(pred[vn]-gt_mm[vn])/(gt_mm[vn]+1e-8))),4) if vn.sum()>0 else np.nan,
    }

print(f"Modelos cargados : {list(DEPTH_MODELS.keys())}")
if not monovit_ok:
    print("⚠️  MonoViT ausente — el experimento correrá con 2 modelos")
print(f"Enhancements     : {list(CORRECTIONS.keys())}")
print(f"Total eval       : {len(DEPTH_MODELS)*len(CORRECTIONS)*len(EVAL_KEYFRAMES)}")

---
## 6. Visualización comparativa de los métodos de enhancement

Antes de correr el experimento completo, comparamos visualmente los tres métodos sobre el keyframe de prueba. Esto permite verificar que:

1. EndoLMSPEC corrige correctamente la exposición
2. Retinex elimina el gradiente de iluminación
3. Ambos preservan la textura del tejido

---
## 7. Experimento factorial: 4 enhancements × 3 modelos × 10 keyframes

Total: **120 evaluaciones**.

In [ ]:
import pandas as pd
import numpy as np
import time
from tqdm.notebook import tqdm

# Verificar que los 3 modelos están cargados antes de correr
assert "MonoViT" in DEPTH_MODELS, "MonoViT no está en DEPTH_MODELS — corre cell-18 primero"
assert len(DEPTH_MODELS) == 3, f"Se esperan 3 modelos, hay {len(DEPTH_MODELS)}"
print(f"✓ Modelos: {list(DEPTH_MODELS.keys())}")
print(f"✓ Enhancements: {list(CORRECTIONS.keys())}")
print(f"✓ Total evaluaciones: {len(DEPTH_MODELS)*len(CORRECTIONS)*len(EVAL_KEYFRAMES)}")

results = []

for dataset_id, keyframe_id in tqdm(EVAL_KEYFRAMES, desc="Keyframes"):
    img_rgb, gt_mm = load_keyframe(SCARED_ROOT, dataset_id, keyframe_id)

    for corr_name, corr_fn in CORRECTIONS.items():
        t_corr0 = time.perf_counter()
        img_corr = corr_fn(img_rgb)
        t_corr_ms = (time.perf_counter() - t_corr0) * 1000

        for model_name, depth_fn in DEPTH_MODELS.items():
            times_inf = []
            for _ in range(3):
                depth_rel, t_ms = depth_fn(img_corr)
                times_inf.append(t_ms)
            t_inf_ms = float(np.median(times_inf))

            metrics = compute_metrics(img_rgb, img_corr, depth_rel, gt_mm, CAP_MM)

            results.append({
                "Dataset":      dataset_id,
                "Keyframe":     keyframe_id,
                "Modelo":       model_name,
                "Método":       corr_name,
                "T_enhance_ms": round(t_corr_ms, 1),
                "T_depth_ms":   round(t_inf_ms, 1),
                "T_total_ms":   round(t_corr_ms + t_inf_ms, 1),
                "FPS":          round(1000 / (t_corr_ms + t_inf_ms), 1),
                **metrics,
            })

            print(f"  {dataset_id}/{keyframe_id} | {model_name:16s} | {corr_name:12s} | "
                  f"AbsRel={metrics['AbsRel']:.4f}  t={t_corr_ms+t_inf_ms:.0f}ms")

df = pd.DataFrame(results)
df.to_csv(OUT_DIR / "avance4_results.csv", index=False)
print(f"\n✓ Experimento completo — {len(df)} evaluaciones")

In [ ]:
import io, zipfile, tifffile, cv2, numpy as np

def load_keyframe(scared_root, dataset_id, keyframe_id):
    zip_path = scared_root / f"{dataset_id}.zip"
    with zipfile.ZipFile(zip_path) as z:
        buf = np.frombuffer(z.read(f"{dataset_id}/{keyframe_id}/Left_Image.png"), np.uint8)
        img = cv2.cvtColor(cv2.imdecode(buf, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        with z.open(f"{dataset_id}/{keyframe_id}/left_depth_map.tiff") as f:
            tiff = tifffile.imread(io.BytesIO(f.read()))
        depth_z = tiff[..., 2].astype(np.float32)
        depth_z[depth_z <= 0] = np.nan
    return img, depth_z

# Diagnóstico
img_test, depth_test = load_keyframe(SCARED_ROOT, *EVAL_KEYFRAMES[0])
print(f"Imagen : {img_test.shape}  GT válido: {(~np.isnan(depth_test)).mean()*100:.1f}%")
print(f"Modelos confirmados: {list(DEPTH_MODELS.keys())}")

In [ ]:
import pandas as pd
import numpy as np

numeric_cols = ["AbsRel","RMSE","Chamfer","PSNR","SSIM",
                "AbsRel_spec","AbsRel_nospec","T_total_ms","FPS"]

summary = (
    df.groupby(["Modelo","Método"])[numeric_cols]
    .mean().round(4)
    .sort_values(["Modelo","AbsRel"])
)

# Mejora vs baseline (none) por modelo
for model in df["Modelo"].unique():
    sub = summary.loc[model]
    if "none" in sub.index:
        baseline_absrel = sub.loc["none", "AbsRel"]
        baseline_spec   = sub.loc["none", "AbsRel_spec"]
        summary.loc[(model, slice(None)), "ΔAbsRel_%"]      = (
            (summary.loc[(model, slice(None)), "AbsRel"] - baseline_absrel)
            / baseline_absrel * 100).round(1)
        summary.loc[(model, slice(None)), "ΔAbsRel_spec_%"] = (
            (summary.loc[(model, slice(None)), "AbsRel_spec"] - baseline_spec)
            / baseline_spec * 100).round(1)

print("═" * 100)
print("RESUMEN — Promedio 10 keyframes (datasets 8–9)")
print("═" * 100)
for model in df["Modelo"].unique():
    print(f"\n▶ {model}")
    print(summary.loc[model][["AbsRel","RMSE","Chamfer",
                               "AbsRel_spec","ΔAbsRel_%","ΔAbsRel_spec_%","FPS"]].to_string())
print("\nΔ% negativo = mejora vs. baseline (none)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models_list  = list(df["Modelo"].unique())
methods_list = list(df["Método"].unique())
colors = {"none":"#888888", "retinex":"#2766CB", "endolmspec":"#E0A800", "iat":"#2ecc71"}

metrics_plot = [
    ("AbsRel",       "AbsRel (↓ mejor)",       "Geométrica — primaria"),
    ("RMSE",         "RMSE mm (↓ mejor)",        "Geométrica"),
    ("AbsRel_spec",  "AbsRel especular (↓ mejor)","Hipótesis central"),
    ("FPS",          "FPS total (↑ mejor)",      "Viabilidad clínica"),
]

n_metrics = len(metrics_plot)
n_models  = len(models_list)
x = np.arange(len(methods_list))
bar_w = 0.6

fig, axes = plt.subplots(n_models, n_metrics,
                         figsize=(5 * n_metrics, 4.5 * n_models),
                         sharey="col")   # mismo eje Y por columna (métrica)

fig.suptitle("Evaluación: Image Enhancement × Modelos de Depth en SCARED\n"
             "(promedio 10 keyframes, datasets 8–9)",
             fontsize=14, fontweight="bold", y=1.01)

for row, model in enumerate(models_list):
    sub = summary.loc[model] if model in summary.index.get_level_values(0) else None

    for col, (metric, label, cat) in enumerate(metrics_plot):
        ax = axes[row, col] if n_models > 1 else axes[col]

        vals = []
        for m in methods_list:
            if sub is not None and m in sub.index:
                vals.append(sub.loc[m, metric])
            else:
                vals.append(np.nan)

        bar_colors = [colors.get(m, "#aaa") for m in methods_list]
        bars = ax.bar(x, vals, width=bar_w, color=bar_colors, edgecolor="white", linewidth=0.8)

        ax.set_xticks(x)
        ax.set_xticklabels(methods_list, fontsize=11)
        ax.grid(axis="y", alpha=0.3)

        # Título de columna solo en primera fila
        if row == 0:
            ax.set_title(f"{label}\n[{cat}]", fontsize=10, pad=6)

        # Etiqueta de modelo solo en primera columna
        if col == 0:
            ax.set_ylabel(model, fontsize=12, fontweight="bold", labelpad=8)

        # Valores encima de cada barra
        for bar, val in zip(bars, vals):
            if not np.isnan(val):
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + ax.get_ylim()[1] * 0.01,
                        f"{val:.3f}",
                        ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(OUT_DIR / "avance4_metricas_comparativa.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 11. Conclusiones

### Hallazgos principales

| Modelo | Método | AbsRel | RMSE (mm) | AbsRel_spec | Δ AbsRel | FPS |
|---|---|---|---|---|---|---|
| **Endo-Depth** | none | 0.165 | 12.47 | 0.209 | — | 36.6 |
| **Endo-Depth** | retinex | 0.153 | 11.58 | 0.195 | **−7.2%** | 2.8 |
| **Endo-Depth** | endolmspec | 0.164 | 12.95 | 0.203 | −1.0% | 8.0 |
| **Endo-Depth** | iat | **0.145** | **11.07** | **0.184** | **−12.0%** | 4.4 |
| **EndoSfMLearner** | none | 0.291 | 21.00 | 0.320 | — | 158.9 |
| **EndoSfMLearner** | iat | 0.290 | 20.97 | 0.320 | −0.1% | 4.8 |
| **MonoViT** | none | 0.305 | 23.73 | 0.304 | — | 18.9 |
| **MonoViT** | iat | **0.283** | 21.34 | 0.288 | **−7.3%** | 4.0 |
| **MonoViT** | retinex | 0.314 | 26.57 | **0.275** | +3.1% | 2.6 |

### Interpretación por modelo

#### Endo-Depth — IAT mejora consistentemente
IAT es el mejor enhancement para Endo-Depth en todas las métricas: **−12.0% AbsRel**, −12.1% AbsRel_spec, y −11.2% RMSE. La mejora en zona especular (−12.1%) es casi igual a la mejora global (−12.0%), lo que indica que IAT no actúa específicamente en especulares sino que mejora la imagen de forma global. Retinex también mejora (−7.2%) con un comportamiento más selectivo. EndoLMSPEC prácticamente no ayuda a Endo-Depth (−1.0%) — posible domain gap con las imágenes SCARED.

#### EndoSfMLearner — prácticamente insensible al enhancement
Todos los métodos dan mejoras menores al 0.1%. EndoSfMLearner fue entrenado con **brightness-aware photometric loss**, lo que ya lo hace robusto a variaciones de iluminación por diseño. El enhancement no aporta información nueva que el modelo no haya aprendido a ignorar durante el entrenamiento.

#### MonoViT — IAT ayuda, Retinex y EndoLMSPEC perjudican
IAT mejora AbsRel en −7.3% pero Retinex (+3.1%) y EndoLMSPEC (+8.5%) **empeoran** el resultado. Esto es esperado: MonoViT fue entrenado en KITTI (imágenes outdoor naturales), y Retinex/EndoLMSPEC al modificar agresivamente el color y la iluminación producen imágenes fuera de la distribución de entrenamiento del modelo. IAT, con su corrección más suave (local mul/add + gamma), preserva mejor la distribución que MonoViT espera.

Curiosamente, Retinex **mejora AbsRel_spec de MonoViT** (−9.8% en zona especular) aunque empeora el AbsRel global — sugiere que los especulares son el punto débil de MonoViT y Retinex los atenúa, pero introduce ruido en el resto de la imagen.

### Conclusión general

**IAT/EndoViT es el mejor enhancement en 2 de 3 modelos** (Endo-Depth y MonoViT). Para EndoSfMLearner el enhancement no tiene impacto significativo. La hipótesis del proyecto se **confirma parcialmente**: la corrección de iluminación reduce el error de profundidad, pero el efecto depende fuertemente del modelo — modelos con brightness-aware loss son inmunes, modelos sin ella se benefician de IAT.

**Para uso clínico**: ningún método de enhancement supera el umbral de 25 FPS combinado con un modelo de depth pesado. IAT con Endo-Depth (4.4 FPS total) y con MonoViT (4.0 FPS) son los mejores pero insuficientes para tiempo real sin optimización (TensorRT, FP16, resolución reducida).

## Referencias

- Zhao, C., et al. (2022). MonoViT: Self-Supervised Monocular Depth Estimation with a Vision Transformer. *3DV 2022*. arXiv:2208.03543.
- Ozyoruk, K. B., et al. (2020). EndoSLAM Dataset and Endo-SfMLearner. *arXiv:2006.16670*.
- García-Vega, A., et al. (2022). Multi-Scale Structural-aware Exposure Correction for Endoscopic Imaging. *arXiv:2210.15033*.
- Wang, T., et al. (2022). Ultra-High-Definition Low-Light Image Enhancement. *AAAI 2022*.
- Recasens, D., et al. (2021). Endo-Depth-and-Motion. *arXiv:2103.16525*.
- Rahman, Z., et al. (2004). Retinex processing for automatic image enhancement. *Journal of Electronic Imaging*, 13(1). https://doi.org/10.1117/1.1636183
- Allan, M., et al. (2021). SCARED Challenge. *arXiv:2101.01133*.

In [ ]:
import subprocess, shutil
from pathlib import Path

# ── Clonar repo si no existe ──────────────────────────────────────────────────
REPO_URL  = "https://github.com/jmtoral/proyecto_integrador_52.git"
REPO_DIR  = Path("/content/proyecto_integrador_52")

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

# Configurar identidad git
subprocess.check_call(["git", "-C", str(REPO_DIR), "config", "user.email", "jmtoralcruz@gmail.com"])
subprocess.check_call(["git", "-C", str(REPO_DIR), "config", "user.name",  "jmtoral"])

# Pull para estar al día
subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--rebase"])

# ── Copiar imágenes generadas → outcomes/avance4/ ─────────────────────────────
dest = REPO_DIR / "outcomes" / "avance4"
dest.mkdir(parents=True, exist_ok=True)

imgs = [
    "avance4_metricas_comparativa.png",
    "avance4_robustez_especular.png",
    "avance4_enhancement_comparativa.png",
    "avance4_results.csv",
]
copied = []
for name in imgs:
    src = OUT_DIR / name
    if src.exists():
        shutil.copy(src, dest / name)
        copied.append(name)
        print(f"  ✓ {name}")
    else:
        print(f"  ✗ {name} — no encontrado en {OUT_DIR}")

# También copiar visualizaciones cualitativas
for f in OUT_DIR.glob("avance4_viz_*.png"):
    shutil.copy(f, dest / f.name)
    copied.append(f.name)
    print(f"  ✓ {f.name}")

# ── Commit y push ─────────────────────────────────────────────────────────────
subprocess.check_call(["git", "-C", str(REPO_DIR), "add", "outcomes/avance4/"])
result = subprocess.run(
    ["git", "-C", str(REPO_DIR), "diff", "--cached", "--name-only"],
    capture_output=True, text=True)
if result.stdout.strip():
    subprocess.check_call([
        "git", "-C", str(REPO_DIR), "commit",
        "-m", f"Add: imágenes Avance4 ImageEnhancement ({len(copied)} archivos)"
    ])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "push"])
    print(f"\n✓ Push exitoso — {len(copied)} archivos en outcomes/avance4/")
else:
    print("Sin cambios nuevos que commitear")

---
## 12. Exportar imágenes al repositorio GitHub

Clona el repo, copia las imágenes generadas a `outcomes/avance4/` y hace push para que estén disponibles en el sitio web.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

methods_list = list(df["Método"].unique())
models_list  = list(df["Modelo"].unique())
x = np.arange(len(methods_list))
w = 0.3

# Un panel por modelo × 2 vistas (keyframe_0 y promedio)
n_cols = 2
n_rows = len(models_list)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows), sharey="row")
if n_rows == 1:
    axes = axes[np.newaxis, :]

fig.suptitle("Robustez a especulares: AbsRel en zona especular vs. tejido normal",
             fontsize=13, fontweight="bold")

panels = [
    (EVAL_KEYFRAMES[0][0], EVAL_KEYFRAMES[0][1]),
    (None, None),
]
panel_titles = [f"{EVAL_KEYFRAMES[0][0]}/{EVAL_KEYFRAMES[0][1]}", "Promedio — todos los keyframes"]

for row, model in enumerate(models_list):
    for col, ((ds_id, kf_id), ptitle) in enumerate(zip(panels, panel_titles)):
        ax = axes[row, col]

        if ds_id is not None:
            sub = df[(df["Modelo"]==model) & (df["Dataset"]==ds_id) & (df["Keyframe"]==kf_id)]
        else:
            sub = df[df["Modelo"]==model]

        spec_vals   = [sub[sub["Método"]==m]["AbsRel_spec"].mean()   for m in methods_list]
        nospec_vals = [sub[sub["Método"]==m]["AbsRel_nospec"].mean() for m in methods_list]

        b1 = ax.bar(x - w/2, spec_vals,   w, label="Zona especular",  color="#e74c3c", alpha=0.85)
        b2 = ax.bar(x + w/2, nospec_vals, w, label="Tejido normal",   color="#2766CB", alpha=0.85)

        ax.set_xticks(x)
        ax.set_xticklabels(methods_list, fontsize=11)
        ax.set_ylabel("AbsRel")
        ax.set_title(f"{model} — {ptitle}", fontsize=10)
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.3)

        for bars, vals in [(b1, spec_vals), (b2, nospec_vals)]:
            for bar, val in zip(bars, vals):
                if not np.isnan(val):
                    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                            f"{val:.3f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / "avance4_robustez_especular.png", dpi=150, bbox_inches="tight")
plt.show()

# Interpretación cuantitativa por modelo
for model in models_list:
    sub_none = df[(df["Modelo"]==model) & (df["Método"]=="none")]
    if sub_none.empty:
        continue
    b_spec = sub_none["AbsRel_spec"].mean()
    b_nosp = sub_none["AbsRel_nospec"].mean()
    print(f"\n▶ {model} — reducción vs. baseline (none):")
    for m in methods_list:
        if m == "none": continue
        sub_m = df[(df["Modelo"]==model) & (df["Método"]==m)]
        s = sub_m["AbsRel_spec"].mean()
        n = sub_m["AbsRel_nospec"].mean()
        print(f"  {m:12s}: spec {s:.4f} ({(b_spec-s)/b_spec*100:+.1f}%)  "
              f"nospec {n:.4f} ({(b_nosp-n)/b_nosp*100:+.1f}%)")

---
## 11. Conclusiones

### Hallazgos principales *(se completa después de correr)*

| Modelo | Método | AbsRel | RMSE | AbsRel_spec | FPS |
|---|---|---|---|---|---|
| Endo-Depth | none/retinex/endolmspec/iat | — | — | — | — |
| EndoSfMLearner | none/retinex/endolmspec/iat | — | — | — | — |
| MonoViT | none/retinex/endolmspec/iat | — | — | — | — |

### Comparativa de modelos de depth

| | **Endo-Depth** | **EndoSfMLearner** | **MonoViT** |
|---|---|---|---|
| **Arquitectura** | ResNet18 + DepthDecoder | DispResNet18 | MPViT-Small + HR-Depth |
| **Encoder** | CNN | CNN | Vision Transformer |
| **Entrenamiento** | Hamlyn (laparoscopy) | EndoSLAM (capsule) | KITTI (outdoor) |
| **Brightness-aware** | No | Sí | No |

### Comparativa de enhancements

| | **Retinex** | **EndoLMSPEC** | **IAT/EndoViT** |
|---|---|---|---|
| **Tipo** | Clásico | U-Net + Laplaciana | Transformer |
| **Entrenamiento** | No | Endo4IE | Endo4IE |
| **Parámetros** | 0 | ~2M | ~90K |

## Referencias

- Zhao, C., et al. (2022). MonoViT: Self-Supervised Monocular Depth Estimation with a Vision Transformer. *3DV 2022*. arXiv:2208.03543.
- Ozyoruk, K. B., et al. (2020). EndoSLAM Dataset and Endo-SfMLearner. *arXiv:2006.16670*.
- García-Vega, A., et al. (2022). Multi-Scale Structural-aware Exposure Correction for Endoscopic Imaging. *arXiv:2210.15033*.
- Wang, T., et al. (2022). Ultra-High-Definition Low-Light Image Enhancement. *AAAI 2022*.
- Recasens, D., et al. (2021). Endo-Depth-and-Motion. *arXiv:2103.16525*.
- Rahman, Z., et al. (2004). Retinex processing for automatic image enhancement. *Journal of Electronic Imaging*, 13(1). https://doi.org/10.1117/1.1636183
- Allan, M., et al. (2021). SCARED Challenge. *arXiv:2101.01133*.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Celda de robustez ya está en cell-24 (la correcta con multi-modelo)
# Esta celda agrega visualización cualitativa por modelo de depth

VIZ_KEYFRAMES = [EVAL_KEYFRAMES[0], EVAL_KEYFRAMES[5]]

for ds_id, kf_id in VIZ_KEYFRAMES:
    img_rgb, gt_mm = load_keyframe(SCARED_ROOT, ds_id, kf_id)
    vmax = np.nanpercentile(gt_mm, 98)
    n_corr = len(CORRECTIONS)
    n_models = len(DEPTH_MODELS)

    fig, axes = plt.subplots(n_models * 2, n_corr + 1,
                             figsize=(5*(n_corr+1), 5*n_models))

    for row_m, (model_name, depth_fn) in enumerate(DEPTH_MODELS.items()):
        row_depth = row_m * 2
        row_err   = row_m * 2 + 1

        for col, (corr_name, corr_fn) in enumerate(CORRECTIONS.items()):
            img_c = corr_fn(img_rgb)
            depth_rel, _ = depth_fn(img_c)
            valid = (~np.isnan(gt_mm)) & (gt_mm > 0) & (gt_mm < CAP_MM)
            scale = np.median(gt_mm[valid]) / (np.median(depth_rel[valid]) + 1e-8)
            pred_mm = depth_rel * scale

            m = compute_metrics(img_rgb, img_c, depth_rel, gt_mm, CAP_MM)

            im = axes[row_depth, col].imshow(pred_mm, cmap="magma_r", vmin=0, vmax=vmax)
            axes[row_depth, col].set_title(
                f"{model_name}\n{corr_name}\nAbsRel={m['AbsRel']:.4f}", fontsize=8)
            axes[row_depth, col].axis("off")

            err = np.abs(pred_mm - gt_mm)
            err[~valid] = np.nan
            axes[row_err, col].imshow(err, cmap="hot", vmin=0,
                                      vmax=np.nanpercentile(err, 95))
            axes[row_err, col].set_title(
                f"spec={m['AbsRel_spec']:.3f}", fontsize=8)
            axes[row_err, col].axis("off")

        # Columna GT
        axes[row_depth, -1].imshow(gt_mm, cmap="magma_r", vmin=0, vmax=vmax)
        axes[row_depth, -1].set_title("GT (mm)", fontsize=8)
        axes[row_depth, -1].axis("off")
        axes[row_err, -1].imshow(specular_mask(img_rgb), cmap="Reds")
        axes[row_err, -1].set_title("Especulares", fontsize=8)
        axes[row_err, -1].axis("off")

    plt.suptitle(f"Depth maps por modelo y enhancement — {ds_id}/{kf_id}",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"avance4_viz_{ds_id}_{kf_id}.png",
                dpi=120, bbox_inches="tight")
    plt.show()